# Featune：从数据到可审阅的特征搜索

本教程用仓库自带的 Adult Income 样本完成一次分类实验。按顺序运行各单元格，你会看到：**数据与外层测试集 → Schema → baseline 与特征搜索 → 试验解释 → 测试集评估 → 导出 → 可选 LLM**。

从仓库根目录先安装 `pip install -e '.[dev]'`。默认模型为本地 TabPFN v2，首次拟合需要下载已固定的权重；前六步不需要 LLM 密钥。示例只用于演示预测流程，字段描述本身不能证明不存在业务时点泄漏。

## 1 · 读取数据，并先留出测试集

`X` 必须是列名唯一的 DataFrame；`y` 是一维标签。先做分层切分，只把 `train_x/train_y` 交给 `study.optimize`。`test_x/test_y` 留到搜索结束后评估，避免用测试结果选择特征。代码兼容从仓库根目录或 `examples/` 启动 Jupyter。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import featune

# Works when Jupyter starts in the repository root or in examples/.
csv_path = next(
    (p for p in [Path("examples/data/adult_income.csv"), Path("data/adult_income.csv")] if p.exists()), None
)
if csv_path is None:
    raise FileNotFoundError("Start Jupyter from the repository root or examples/ directory")
repo_root = csv_path.resolve().parents[2]
data = pd.read_csv(csv_path).sample(n=400, random_state=42)
X = data.drop(columns="income_over_50k")
y = data["income_over_50k"]
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(f"Rows: {len(data)} | search rows: {len(train_x)} | untouched test rows: {len(test_x)}")
print("Target distribution:", y.value_counts().to_dict())
display(train_x.head(3))

## 2 · 用 Schema 说明字段和预测任务

`DatasetSchema` 给搜索器提供可用字段、任务目标、标签含义和预测时点；每个 `FieldSchema` 指明字段名、类型和业务含义。这里所有输入是数值列，其中 0/1 指示列通过 `encoding` 说明编码。目标列已经从 `X` 移除。`available_at` 只是描述，真实可用时点仍需数据负责人核实。

In [ ]:
ENCODING = "1 表示是，0 表示否"
AVAILABLE_AT = "人口普查访谈时"
OBJECTIVE = "根据访谈时已记录字段估计年收入是否超过 50K"
TARGET_DEFINITION = "1 表示年收入超过 50K；0 表示不超过 50K"
PREDICTION_POINT = "人口普查访谈时，所列字段已记录之后"

descriptions = {
    "age": "年龄（岁）",
    "fnlwgt": "人口普查样本权重",
    "education_num": "教育程度数字编码，值越高表示教育程度越高",
    "capital_gain": "申报的年度资本收益",
    "capital_loss": "申报的年度资本损失",
    "hours_per_week": "每周通常工作小时数",
    "workclass_Private": "是否在私营机构工作",
    "education_Bachelors": "最高学历是否为学士",
    "education_Masters": "最高学历是否为硕士",
    "marital_status_Married_civ_spouse": "是否已婚且配偶为平民",
    "occupation_Exec_managerial": "职业是否属于管理岗位",
    "occupation_Prof_specialty": "职业是否属于专业技术岗位",
    "relationship_Husband": "家庭关系是否标注为丈夫",
    "sex_Male": "性别是否标注为男性",
    "native_country_United_States": "原籍国是否为美国",
    "race_White": "种族是否标注为白人",
}
schema = featune.DatasetSchema(
    fields=[
        featune.FieldSchema(
            name=name,
            dtype="numeric",
            description=description,
            encoding=ENCODING
            if name
            not in {"age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"}
            else None,
            available_at=AVAILABLE_AT,
        )
        for name, description in descriptions.items()
    ],
    objective=OBJECTIVE,
    target_definition=TARGET_DEFINITION,
    prediction_point=PREDICTION_POINT,
)
schema.validate_frame(train_x)
print(f"Usable fields: {len(schema.usable)} | target excluded: {'income_over_50k' not in schema.usable}")

### 先看一列受控特征长什么样

`FeatureSpec` 只能使用白名单算子（这里是 `divide`）、已有输入列和合法参数；`Hypothesis` 保存待检验的理由。`FeatureCompiler.fit(train_x)` 只在训练行学习可能有状态的变换，`transform` 复用它们。下面只预览训练行的计算结果，**不会**把这列强行加入最佳模型；正式搜索的每个候选仍会独立验证并做 CV。

In [ ]:
preview_spec = featune.FeatureSpec(
    name="hours_per_age",
    op="divide",
    inputs=["hours_per_week", "age"],
    hypothesis=featune.Hypothesis(
        concept="work intensity",
        rationale="Working hours relative to age may carry predictive information",
        expected_direction="unknown",
    ),
)
compiler = featune.FeatureCompiler(schema, [preview_spec]).fit(train_x)
display(compiler.transform(train_x).head(3)[["hours_per_week", "age", "hours_per_age"]])

## 3 · 评估 baseline，再搜索两个候选

`create_study` 配置主指标、采样器、搜索策略和资源上限。`RandomSampler` 只**提出**封闭 DSL 特征；`CVEvaluator` 在相同的训练折中编译、拟合并评分。`n_trials=2` 表示最多两次候选尝试，baseline 是额外的 trial 0。`max_model_fits=7` 包含 2 次 baseline、最多 4 次候选和 1 次最终 refit。候选较差时，baseline 仍可获胜。

In [ ]:
study = featune.create_study(
    metric="auc",  # higher is better; baseline remains eligible
    sampler=featune.RandomSampler(seed=42),
    search_strategy="greedy",  # extend the best completed feature set
    max_features=2,  # at most two derived columns
    budget=featune.SearchBudget(max_model_fits=7),
)
evaluator = featune.CVEvaluator(cv=2, importance_repeats=0)
study.optimize(train_x, train_y, schema, evaluator=evaluator, n_trials=2)
print(f"Baseline inner-CV AUC: {study.baseline_trial.value:.4f}")
print(f"Best inner-CV AUC:     {study.best_value:.4f}")
print(f"Best trial: {study.best_trial.number} | derived columns: {len(study.best_features)}")

## 4 · 阅读试验，而不只看最高分

`value` 是内层 CV AUC；`delta` 是候选减 baseline，`parent_delta` 是候选减父集合。AUC 越高越好，所以正差值表示改善。`feature_history()` 给出特征算子、输入与假设状态；关闭置换重要性时，部分假设证据会是 `inconclusive`，不能把它当作因果结论。`search_summary()` 汇总实际拟合与资源消耗。

In [ ]:
trials = study.trials_dataframe()
display(
    trials[["number", "parent", "state", "value", "delta", "parent_delta", "feature_count", "model_fits"]]
)
display(
    study.feature_history()[["trial", "feature", "op", "inputs", "status", "reason"]]
    if study.best_features
    else study.feature_history()
)
print("Resource summary:", study.search_summary())

## 5 · 最后才看未参与搜索的测试集

`export_pipeline()` 返回在全部**搜索训练行**上重新拟合的最佳特征编译器和模型。测试 AUC 是一次独立评估；不要根据它反复调参后仍把它称为独立测试。内层 CV 分数和测试分数来自不同数据，不能直接把二者差值当作特征贡献。

In [ ]:
pipeline = study.export_pipeline()  # best feature compiler + fitted model
test_probability = pipeline.predict_proba(test_x)[:, 1]
test_auc = roc_auc_score(test_y, test_probability)
print(f"Untouched test AUC: {test_auc:.4f}")
display(pd.DataFrame({"actual": test_y.to_numpy()[:5], "probability_of_1": test_probability[:5]}))

## 6 · 保存可审阅结果

`features.json` 是选中特征的受控 DSL，`pipeline.joblib` 是可直接 `predict/predict_proba` 的拟合对象，`report.html` 包含交互式试验轨迹。运行产物放在被 Git 忽略的 `runs/`。**joblib 模型可能包含训练数据上下文，只加载可信文件，也不要把模型文件当作脱敏结果公开。**

In [ ]:
output = repo_root / "runs" / "notebook-zh"
output.mkdir(parents=True, exist_ok=True)
study.export_features(output / "features.json")
study.export_pipeline(output / "pipeline.joblib")
study.report(output / "report.html")
print("Saved reviewable results to:", output)

## 7 · 可选：用 LLM 提议一列，再经过同样的 CV

先将项目 `.env` 中的 `FEATUNE_API_KEY`、`FEATUNE_BASE_URL`、`FEATUNE_MODEL` 加载到当前 Jupyter 进程，例如从仓库根目录运行 `set -a; source .env; set +a; jupyter notebook`；可选 `FEATUNE_PROVIDER=anthropic|openai|local`，未设时使用 Anthropic Messages 格式。要实际发送请求，把下一格的 `RUN_LLM` 改为 `True`；默认跳过以避免意外费用。LLM 只提议 JSON 特征，不执行代码，也不直接决定最佳模型；本示例限制为一次提议、最多一列和 30,000 个已知 token。

接口对应关系：`anthropic` 使用 `/v1/messages`；`openai` 使用 `/chat/completions` 和默认的 `max_completion_tokens`；`local` 使用 OpenAI 兼容格式和 `max_tokens`。旧式网关可在 `LLMSampler(..., output_token_parameter="max_tokens")` 中覆盖参数名。温度默认不发送，只有显式设置才发送。

In [ ]:
import os

RUN_LLM = False  # Change to True only when you want to send a billable request.
required = ("FEATUNE_API_KEY", "FEATUNE_BASE_URL", "FEATUNE_MODEL")
missing = [name for name in required if not os.getenv(name)]
if not RUN_LLM:
    print("LLM run skipped; set RUN_LLM=True to opt in.")
elif missing:
    print("LLM run skipped; set these environment variables:", ", ".join(missing))
else:
    llm_sampler = featune.LLMSampler(
        provider=os.getenv("FEATUNE_PROVIDER", "anthropic"),
        model=os.environ["FEATUNE_MODEL"],
        base_url=os.environ["FEATUNE_BASE_URL"],
        max_tokens=1024,
        features_per_trial=1,
    )
    llm_study = featune.create_study(
        metric="auc",
        sampler=llm_sampler,
        search_strategy="greedy",
        max_features=1,
        budget=featune.SearchBudget(max_model_fits=4, max_llm_tokens=30_000),
    )
    llm_study.optimize(
        train_x,
        train_y,
        schema,
        evaluator=featune.CVEvaluator(cv=2),
        n_trials=1,
        refit=False,
    )
    display(llm_study.proposal_history())
    print("LLM trial state:", llm_study.trials[-1].state)
    print("Best inner-CV AUC:", round(llm_study.best_value, 4))

## 下一步

- 将自己的 DataFrame、标签和字段定义替换进步骤 1–2，先核实字段在预测时点是否可用。
- 改用 `search_strategy="autonomous"` 可逐列增删；`max_features` 是上限，重复或无增益候选会保留在 trial 记录中。
- 回归任务用 `metric="rmse"`，默认模型会换为 `TabPFNRegressor`；损失指标的负 `delta` 才是改善。
- 进阶参数、分组/时间 CV、预算和恢复见 [中文指南](../docs/zh/guide.md) 与 [API 参考](../docs/zh/reference.md)。